## Analysis Objective

The objective of this analysis is to evaluate retail performance from an omnichannel business perspective.

Rather than analyzing overall sales in isolation, the focus is on channel-driven profitability, returns, customer behavior, and operational insights that can support business decision-making.

### Phase A: Data Loading & Validation

- Purpose: Ensure cleaned data is ready for analysis

- Load cleaned dataset

- Validate schema and row counts

- Confirm presence of engineered features

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

In [2]:
df = pd.read_csv("/Users/hepigediya/Desktop/retail-omnichannel-analytics/data/processed/cleaned_retail_sales.csv")

In [3]:
df.shape

(1027017, 17)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1027017 entries, 0 to 1027016
Data columns (total 17 columns):
 #   Column                 Non-Null Count    Dtype  
---  ------                 --------------    -----  
 0   invoice_no             1027017 non-null  object 
 1   stock_code             1027017 non-null  object 
 2   description            1027017 non-null  object 
 3   quantity               1027017 non-null  int64  
 4   invoice_date           1027017 non-null  object 
 5   unit_price             1027017 non-null  float64
 6   customer_id            797815 non-null   float64
 7   country                1027017 non-null  object 
 8   is_return              1027017 non-null  bool   
 9   is_cancellation        1027017 non-null  bool   
 10  is_anonymous_customer  1027017 non-null  bool   
 11  sales_amount           1027017 non-null  float64
 12  order_date             1027017 non-null  object 
 13  order_month            1027017 non-null  int64  
 14  order_year        

In [5]:
df['invoice_date'] = pd.to_datetime(df['invoice_date'], errors='coerce')

In [6]:
df['invoice_date'].dtype

dtype('<M8[ns]')

In [7]:
df.head()

,invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country,is_return,is_cancellation,is_anonymous_customer,sales_amount,order_date,order_month,order_year,is_bulk_transaction,sales_channel
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,False,False,False,83.4,2009-12-01,12,2009,False,NaN
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,False,False,False,81.0,2009-12-01,12,2009,False,NaN
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,False,False,False,81.0,2009-12-01,12,2009,False,NaN
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,False,False,False,100.8,2009-12-01,12,2009,False,NaN
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,False,False,False,30.0,2009-12-01,12,2009,False,NaN


 ### Phase B: Channel Derivation (Core Step)

- Purpose: Establish omnichannel structure

- Derive sales_channel using invoice-level behavior

- Document assumptions and thresholds

- Validate channel distribution

- This phase defines the analytical foundation of the project.

In [8]:
""" Aggregate at Invoice Level
first move from line-item level → invoice level. """

invoice_summary = (
    df.groupby('invoice_no')
      .agg(
          total_quantity=('quantity', 'sum'),
          total_sales=('sales_amount', 'sum')
      )
      .reset_index()
)


In [9]:
# Channel derivation rule
invoice_summary['derived_channel'] = np.where(
    invoice_summary['total_quantity'] >= 20,
    'Offline',
    'Online'
)

In [10]:
invoice_summary['derived_channel'].value_counts()

derived_channel
Offline    35929
Online     12440
Name: count, dtype: int64

In [11]:
df = df.merge(
    invoice_summary[['invoice_no', 'derived_channel']],
    on='invoice_no',
    how='left'
)

In [12]:
df = df.rename(columns={'derived_channel': 'sales_channel'})

In [13]:
# Remove placeholder channel column from cleaning phase
df = df.drop(columns=['sales_channel'])

In [14]:
df = df.merge(
    invoice_summary[['invoice_no', 'derived_channel']],
    on='invoice_no',
    how='left'
)

df = df.rename(columns={'derived_channel': 'sales_channel'})

In [15]:
df['sales_channel'].value_counts()

sales_channel
Offline    996526
Online      30491
Name: count, dtype: int64

### Phase C: Channel-Level Revenue Analysis

- Purpose: Understand where money is coming from

- Total revenue by channel

- Revenue contribution percentage

- Revenue trend comparison by channel

- Business question:

    Which channel drives the majority of sales, and how stable is it?

In [16]:
revenue_by_channel = (
    df.groupby('sales_channel')['sales_amount']
      .sum()
      .sort_values(ascending=False)
)

revenue_by_channel

sales_channel
Offline    1.993788e+07
Online    -9.236721e+05
Name: sales_amount, dtype: float64

In [17]:
revenue_pct = (
    revenue_by_channel / revenue_by_channel.sum() * 100
).round(2)

revenue_pct

sales_channel
Offline    104.86
Online      -4.86
Name: sales_amount, dtype: float64

In [18]:
orders_by_channel = (
    df.groupby('sales_channel')['invoice_no']
      .nunique()
)

orders_by_channel

sales_channel
Offline    35929
Online     12440
Name: invoice_no, dtype: int64

In [19]:
aov_by_channel = (
    revenue_by_channel / orders_by_channel
).round(2)

aov_by_channel

sales_channel
Offline    554.92
Online     -74.25
dtype: float64

In [20]:
channel_kpis = pd.DataFrame({
    'total_revenue': revenue_by_channel,
    'revenue_pct': revenue_pct,
    'total_orders': orders_by_channel,
    'avg_order_value': aov_by_channel
})

channel_kpis

,total_revenue,revenue_pct,total_orders,avg_order_value
sales_channel,,,,
Offline,1.993788e+07,104.86,35929,554.92
Online,-9.236721e+05,-4.86,12440,-74.25


Which channel drives the majority of sales, and how stable is it?

Answer:

The Offline channel drives the majority of sales and is the more stable revenue source. It contributes almost all of the positive net revenue, supported by a higher number of orders and significantly larger average order values. Offline sales show consistent performance with minimal disruption from returns, making this channel a reliable and predictable driver of business revenue.

In contrast, the Online channel contributes a smaller share of total orders and shows revenue instability due to high return volumes. As a result, online sales do not translate into sustainable net revenue despite transaction activity.

### Phase D: Returns Impact by Channel

- Purpose: Identify profitability risk

- Return rate by channel

- Revenue lost due to returns

- Products with high return concentration by channel

- Business question:

    Which channel is more operationally expensive due to returns?

In [21]:
# Number of returned orders per channel
returned_orders = (
    df[df['is_return']]
    .groupby('sales_channel')['invoice_no']
    .nunique()
)

# Total orders per channel
total_orders = (
    df.groupby('sales_channel')['invoice_no']
    .nunique()
)

# Return rate (%)
return_rate_orders = (
    returned_orders / total_orders * 100
).round(2)

return_rate_orders

sales_channel
Offline      NaN
Online     66.65
Name: invoice_no, dtype: float64

In [22]:
returns_revenue_loss = (
    df[df['is_return']]
    .groupby('sales_channel')['sales_amount']
    .sum()
    .abs()
)

returns_revenue_loss

sales_channel
Online    1462424.18
Name: sales_amount, dtype: float64

In [23]:
high_return_products = (
    df[df['is_return']]
    .groupby(['sales_channel', 'stock_code', 'description'])
    .agg(
        returned_quantity=('quantity', lambda x: x.abs().sum()),
        return_value=('sales_amount', lambda x: x.abs().sum())
    )
    .reset_index()
    .sort_values('return_value', ascending=False)
)

high_return_products.head(10)

,sales_channel,stock_code,description,returned_quantity,return_value
3082,Online,M,Manual,5444,422926.14
3075,Online,AMAZONFEE,AMAZON FEE,33,241988.30
2177,Online,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,168469.60
1918,Online,23166,MEDIUM CERAMIC TOP STORAGE JAR,74494,77479.64
3077,Online,BANK CHARGES,Bank Charges,72,33902.53
1310,Online,22423,REGENCY CAKESTAND 3 TIER,1450,16545.30
3084,Online,POST,POSTAGE,302,15252.01
3080,Online,D,Discount,3064,13277.52
2859,Online,85123A,WHITE HANGING HEART T-LIGHT HOLDER,3637,9387.10
3079,Online,CRUK,CRUK Commission,16,7933.43


In [24]:
""" Top 5 per channel """
top_returns_by_channel = (
    high_return_products
    .groupby('sales_channel')
    .head(5)
)

top_returns_by_channel

,sales_channel,stock_code,description,returned_quantity,return_value
3082,Online,M,Manual,5444,422926.14
3075,Online,AMAZONFEE,AMAZON FEE,33,241988.30
2177,Online,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,168469.60
1918,Online,23166,MEDIUM CERAMIC TOP STORAGE JAR,74494,77479.64
3077,Online,BANK CHARGES,Bank Charges,72,33902.53


In [25]:
returns_risk_summary = pd.DataFrame({
    'total_orders': total_orders,
    'returned_orders': returned_orders,
    'return_rate_pct': return_rate_orders,
    'returns_revenue_loss': returns_revenue_loss
})

returns_risk_summary

,total_orders,returned_orders,return_rate_pct,returns_revenue_loss
sales_channel,,,,
Offline,35929,NaN,NaN,NaN
Online,12440,8291.0,66.65,1462424.18


Which channel is more operationally expensive due to returns?

Answer:

The Online channel is significantly more operationally expensive due to returns. Approximately 66.65% of online orders are returned, resulting in a substantial revenue loss of over 1.46 million. This high return rate indicates increased costs related to reverse logistics, inventory handling, and customer service.

The Offline channel shows negligible return activity, suggesting lower operational burden and better cost efficiency. This makes offline sales operationally more efficient and less risky from a profitability standpoint.

### Phase E: Customer Behavior by Channel

- Purpose: Evaluate customer value

- Number of customers per channel

- Anonymous vs identified customer impact

- Repeat vs one-time behavior (where applicable)

- Business question:

    Which channel produces more valuable and identifiable customers?

In [26]:
customers_by_channel = (
    df[~df['is_anonymous_customer']]
    .groupby('sales_channel')['customer_id']
    .nunique()
)

customers_by_channel

sales_channel
Offline    5754
Online     3037
Name: customer_id, dtype: int64

In [27]:
customer_type_distribution = (
    df.groupby(['sales_channel', 'is_anonymous_customer'])['invoice_no']
    .nunique()
    .reset_index(name='order_count')
)

customer_type_distribution

,sales_channel,is_anonymous_customer,order_count
0,Offline,False,34201
1,Offline,True,1728
2,Online,False,10669
3,Online,True,1771


In [28]:
customer_type_pivot = customer_type_distribution.pivot(
    index='sales_channel',
    columns='is_anonymous_customer',
    values='order_count'
).rename(columns={False: 'identified_orders', True: 'anonymous_orders'})

customer_type_pivot

is_anonymous_customer,identified_orders,anonymous_orders
sales_channel,,
Offline,34201,1728
Online,10669,1771


Repeat customer → more than 1 order

One-time customer → exactly 1 order

In [29]:
customer_order_counts = (
    df[~df['is_anonymous_customer']]
    .groupby(['sales_channel', 'customer_id'])['invoice_no']
    .nunique()
    .reset_index(name='order_count')
)

In [30]:
repeat_behavior = (
    customer_order_counts
    .assign(customer_type=lambda x: np.where(x['order_count'] > 1, 'Repeat', 'One-time'))
    .groupby(['sales_channel', 'customer_type'])['customer_id']
    .nunique()
    .reset_index(name='customer_count')
)

repeat_behavior

,sales_channel,customer_type,customer_count
0,Offline,One-time,1638
1,Offline,Repeat,4116
2,Online,One-time,1283
3,Online,Repeat,1754


In [31]:
repeat_behavior_pivot = repeat_behavior.pivot(
    index='sales_channel',
    columns='customer_type',
    values='customer_count'
)

repeat_behavior_pivot

customer_type,One-time,Repeat
sales_channel,,
Offline,1638,4116
Online,1283,1754


In [32]:
customer_value_summary = pd.DataFrame({
    'identified_customers': customers_by_channel,
    'identified_orders': customer_type_pivot['identified_orders'],
    'anonymous_orders': customer_type_pivot['anonymous_orders'],
    'repeat_customers': repeat_behavior_pivot.get('Repeat'),
    'one_time_customers': repeat_behavior_pivot.get('One-time')
})

customer_value_summary

,identified_customers,identified_orders,anonymous_orders,repeat_customers,one_time_customers
sales_channel,,,,,
Offline,5754,34201,1728,4116,1638
Online,3037,10669,1771,1754,1283


Which channel produces more valuable and identifiable customers?
Answer:

The Offline channel produces more valuable and identifiable customers. Offline sales are dominated by identified customers, enabling better tracking of purchasing behavior and stronger potential for long-term customer relationships. A higher proportion of offline customers exhibit repeat purchasing behavior, indicating greater customer loyalty and lifetime value.

In contrast, the Online channel contains a significantly higher share of anonymous and one-time transactions. This limits customer visibility and reduces opportunities for retention and personalization. Combined with high return rates, online customers currently represent lower realized customer value.

### Phase F: Product Performance by Channel

- Purpose: Optimize assortment and inventory

- Top products by revenue per channel

- High-volume vs high-return products

- Channel-specific product preferences

- Business question:

    Should the same products be pushed across both channels?

In [33]:
top_products_by_revenue = (
    df[df['quantity'] > 0]  # gross sales only
    .groupby(['sales_channel', 'stock_code', 'description'])['sales_amount']
    .sum()
    .reset_index()
    .sort_values(['sales_channel', 'sales_amount'], ascending=[True, False])
)

top_products_by_revenue.groupby('sales_channel').head(10)

,sales_channel,stock_code,description,sales_amount
1888,Offline,22423,REGENCY CAKESTAND 3 TIER,324424.51
5584,Offline,DOT,DOTCOM POSTAGE,307347.59
4978,Offline,85123A,WHITE HANGING HEART T-LIGHT HOLDER,256944.90
3347,Offline,23843,"PAPER CRAFT , LITTLE BIRDIE",168469.60
3708,Offline,47566,PARTY BUNTING,147432.70
4946,Offline,85099B,JUMBO BAG RED RETROSPOT,145830.66
4623,Offline,84879,ASSORTED COLOUR BIRD ORNAMENT,129196.05
1474,Offline,22086,PAPER CHAIN KIT 50'S CHRISTMAS,117568.37
5587,Offline,POST,POSTAGE,103585.75
2806,Offline,23166,MEDIUM CERAMIC TOP STORAGE JAR,81693.42


In [34]:
high_volume_products = (
    df[df['quantity'] > 0]
    .groupby(['sales_channel', 'stock_code', 'description'])['quantity']
    .sum()
    .reset_index()
    .sort_values(['sales_channel', 'quantity'], ascending=[True, False])
)

high_volume_products.groupby('sales_channel').head(10)

,sales_channel,stock_code,description,quantity
4157,Offline,84077,WORLD WAR 2 GLIDERS ASSTD DESIGNS,106122
4978,Offline,85123A,WHITE HANGING HEART T-LIGHT HOLDER,93936
3347,Offline,23843,"PAPER CRAFT , LITTLE BIRDIE",80995
4623,Offline,84879,ASSORTED COLOUR BIRD ORNAMENT,80006
2806,Offline,23166,MEDIUM CERAMIC TOP STORAGE JAR,78027
4946,Offline,85099B,JUMBO BAG RED RETROSPOT,77220
121,Offline,17003,BROCADE RING PURSE,70369
1381,Offline,21977,PACK OF 60 PINK PAISLEY CAKE CASES,56055
4783,Offline,84991,60 TEATIME FAIRY CAKE CASES,54023
1604,Offline,22197,SMALL POPCORN HOLDER,48498


In [35]:
high_return_products = (
    df[df['is_return']]
    .groupby(['sales_channel', 'stock_code', 'description'])
    .agg(
        returned_quantity=('quantity', lambda x: x.abs().sum()),
        return_value=('sales_amount', lambda x: x.abs().sum())
    )
    .reset_index()
    .sort_values(['sales_channel', 'return_value'], ascending=[True, False])
)

high_return_products.groupby('sales_channel').head(10)

,sales_channel,stock_code,description,returned_quantity,return_value
3082,Online,M,Manual,5444,422926.14
3075,Online,AMAZONFEE,AMAZON FEE,33,241988.30
2177,Online,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,168469.60
1918,Online,23166,MEDIUM CERAMIC TOP STORAGE JAR,74494,77479.64
3077,Online,BANK CHARGES,Bank Charges,72,33902.53
1310,Online,22423,REGENCY CAKESTAND 3 TIER,1450,16545.30
3084,Online,POST,POSTAGE,302,15252.01
3080,Online,D,Discount,3064,13277.52
2859,Online,85123A,WHITE HANGING HEART T-LIGHT HOLDER,3637,9387.10
3079,Online,CRUK,CRUK Commission,16,7933.43


In [36]:
top_online_products = set(
    top_products_by_revenue[top_products_by_revenue['sales_channel'] == 'Online']
    .head(20)['stock_code']
)

top_offline_products = set(
    top_products_by_revenue[top_products_by_revenue['sales_channel'] == 'Offline']
    .head(20)['stock_code']
)

common_products = top_online_products.intersection(top_offline_products)

len(common_products), common_products

(5, {'21621', '22423', '47566', 'DOT', 'POST'})

In [37]:
product_strategy_summary = pd.DataFrame({
    'top_revenue_products': top_products_by_revenue.groupby('sales_channel').size(),
    'high_return_products': high_return_products.groupby('sales_channel').size()
})

product_strategy_summary

,top_revenue_products,high_return_products
sales_channel,,
Offline,5597,NaN
Online,3051,3088.0


Should the same products be pushed across both channels?
Answer:

The analysis indicates that the same products should not be uniformly pushed across both channels. Product performance differs significantly by channel. Products that generate high revenue and volume in the Offline channel tend to experience substantially higher return rates when sold Online, leading to profit erosion and increased operational costs.

Offline customers favor bulk and repeat-purchase products, while Online customers show more selective purchasing behavior with higher return sensitivity. As a result, a channel-specific assortment strategy is recommended: prioritize stable, low-return products online and reserve bulk or high-volume items primarily for offline sales.

### Phase G: Time-Based Channel Trends

- Purpose: Detect seasonality and risk

- Monthly revenue trends by channel

- Peak and low periods

- Channel volatility comparison

- Business question:

    Which channel provides more predictable revenue over time?

In [38]:
monthly_channel_revenue = (
    df[df['quantity'] > 0]
    .groupby(['order_year', 'order_month', 'sales_channel'])['sales_amount']
    .sum()
    .reset_index()
)

monthly_channel_revenue

,order_year,order_month,sales_channel,sales_amount
0,2009,12,Offline,814987.370
1,2009,12,Online,7496.580
2,2010,1,Offline,623110.062
3,2010,1,Online,28045.050
4,2010,2,Offline,541501.196
5,2010,2,Online,10377.100
6,2010,3,Offline,768533.541
7,2010,3,Online,62381.720
8,2010,4,Offline,648709.492
9,2010,4,Online,30165.760


In [39]:
peak_low_periods = (
    monthly_channel_revenue
    .groupby('sales_channel')['sales_amount']
    .agg(
        peak_revenue='max',
        lowest_revenue='min',
        avg_revenue='mean'
    )
    .round(2)
)

peak_low_periods

,peak_revenue,lowest_revenue,avg_revenue
sales_channel,,,
Offline,1493889.44,517702.64,797515.28
Online,62381.72,1827.28,21550.08


In [40]:
revenue_volatility = (
    monthly_channel_revenue
    .groupby('sales_channel')['sales_amount']
    .std()
    .round(2)
)

revenue_volatility

sales_channel
Offline    256733.95
Online      16796.16
Name: sales_amount, dtype: float64

In [41]:
avg_monthly_revenue = (
    monthly_channel_revenue
    .groupby('sales_channel')['sales_amount']
    .mean()
)

revenue_cv = (
    revenue_volatility / avg_monthly_revenue
).round(2)

revenue_cv

sales_channel
Offline    0.32
Online     0.78
Name: sales_amount, dtype: float64

In [42]:
time_trend_summary = pd.DataFrame({
    'avg_monthly_revenue': avg_monthly_revenue.round(2),
    'revenue_volatility': revenue_volatility,
    'revenue_cv': revenue_cv,
    'peak_revenue': peak_low_periods['peak_revenue'],
    'lowest_revenue': peak_low_periods['lowest_revenue']
})

time_trend_summary

,avg_monthly_revenue,revenue_volatility,revenue_cv,peak_revenue,lowest_revenue
sales_channel,,,,,
Offline,797515.28,256733.95,0.32,1493889.44,517702.64
Online,21550.08,16796.16,0.78,62381.72,1827.28


Which channel provides more predictable revenue over time?
Answer:

The Offline channel provides more predictable revenue over time. Its monthly revenue shows lower relative volatility and smaller fluctuations between peak and low periods, indicating stable and consistent demand. This makes offline sales more reliable for forecasting, inventory planning, and financial decision-making.

In contrast, the Online channel exhibits higher month-to-month variability and stronger seasonality effects. Combined with elevated return rates, this volatility increases revenue risk and reduces predictability over time.

### Phase H: Geographic Overlay (Optional Enhancement)

- Purpose: Add market-level context

- Country-wise channel performance

- Revenue concentration risk

- Channel dominance by geography

- Business question:

    Does channel performance vary by market?

In [43]:
country_channel_revenue = (
    df[df['quantity'] > 0]
    .groupby(['country', 'sales_channel'])['sales_amount']
    .sum()
    .reset_index()
    .sort_values('sales_amount', ascending=False)
)

country_channel_revenue.head(10)

,country,sales_channel,sales_amount
70,United Kingdom,Offline,1.698967e+07
18,EIRE,Offline,6.303060e+05
44,Netherlands,Offline,5.522650e+05
71,United Kingdom,Online,4.209008e+05
26,Germany,Offline,4.168690e+05
24,France,Offline,3.339369e+05
0,Australia,Offline,1.677401e+05
59,Spain,Offline,1.056648e+05
63,Switzerland,Offline,9.869726e+04
61,Sweden,Offline,8.967059e+04


In [44]:
country_revenue_total = (
    df[df['quantity'] > 0]
    .groupby('country')['sales_amount']
    .sum()
    .sort_values(ascending=False)
)

country_revenue_total.head(10)

country
United Kingdom    1.741057e+07
EIRE              6.587673e+05
Netherlands       5.540381e+05
Germany           4.250197e+05
France            3.504561e+05
Australia         1.692835e+05
Spain             1.083325e+05
Switzerland       1.006856e+05
Sweden            9.186982e+04
Denmark           6.858069e+04
Name: sales_amount, dtype: float64

In [45]:
country_revenue_pct = (
    country_revenue_total / country_revenue_total.sum() * 100
).round(2)

country_revenue_pct.head(10)

country
United Kingdom    85.03
EIRE               3.22
Netherlands        2.71
Germany            2.08
France             1.71
Australia          0.83
Spain              0.53
Switzerland        0.49
Sweden             0.45
Denmark            0.33
Name: sales_amount, dtype: float64

In [46]:
channel_dominance_by_country = (
    country_channel_revenue
    .sort_values(['country', 'sales_amount'], ascending=[True, False])
    .groupby('country')
    .head(1)
    .reset_index(drop=True)
)

channel_dominance_by_country.head(10)

,country,sales_channel,sales_amount
0,Australia,Offline,167740.09
1,Austria,Offline,22993.01
2,Bahrain,Offline,3109.79
3,Belgium,Offline,62460.12
4,Bermuda,Offline,1253.14
5,Brazil,Offline,1411.87
6,Canada,Offline,4332.10
7,Channel Islands,Offline,44271.83
8,Cyprus,Offline,24476.16
9,Czech Republic,Offline,826.74


In [47]:
channel_mix_country = (
    country_channel_revenue
    .pivot(index='country', columns='sales_channel', values='sales_amount')
    .fillna(0)
)

channel_mix_country.head()

sales_channel,Offline,Online
country,,
Australia,167740.09,1543.37
Austria,22993.01,620.00
Bahrain,3109.79,0.00
Belgium,62460.12,2927.70
Bermuda,1253.14,0.00


Does channel performance vary by market?
Answer:

Yes, channel performance varies significantly by market. Revenue is heavily concentrated in a small number of countries, where the Offline channel consistently dominates sales performance. These markets show strong bulk purchasing behavior and more stable revenue patterns.

In contrast, the Online channel contributes unevenly across countries, with limited dominance in most markets and higher exposure to return-related losses. This indicates that channel effectiveness is influenced by regional purchasing behavior, logistics maturity, and customer preferences.

Conclusion:
A uniform omnichannel strategy is not optimal. Channel investment and product assortment should be market-specific, with offline-focused strategies in core revenue countries and carefully optimized online offerings in selective markets.

### Phase I: Business Insights & Recommendations

- Purpose: Convert analysis into action

- Key findings summary

- Channel investment recommendations

- Risk areas and optimization opportunities

Purpose

Translate analytical findings into clear business actions that improve profitability, reduce risk, and support smarter channel and inventory decisions.

Key Findings Summary

1. Offline channel is the primary revenue driver

- Contributes the majority of gross and net revenue

- Higher average order value and bulk purchasing behavior

- More stable and predictable revenue over time

2. Online channel suffers from high return-related losses

- Extremely high return rate compared to offline

- Net revenue is significantly eroded by returns

- High operational cost due to reverse logistics

3. Customer value is stronger in the Offline channel

- Higher proportion of identifiable customers

- Greater repeat-purchase behavior

- Better potential for long-term customer relationships

4. Product performance differs significantly by channel

- Products performing well offline often generate high returns online

- Online channel shows higher sensitivity to product mismatch

5. Channel performance varies by geography

- Revenue is concentrated in a small number of core markets

- Offline channel dominates in high-revenue countries

- Online performance is inconsistent across regions

Channel Investment Recommendations
1. Prioritize Offline Channel Growth

- Continue investing in offline sales where demand is stable

- Focus on bulk-selling products and high AOV categories

- Use offline channel as the backbone for predictable revenue

2. Optimize (Not Scale) Online Channel

- Do not aggressively scale online without fixing return issues

- Shift focus from volume to quality of sales

- Introduce stricter product selection and return controls

Risk Areas Identified

1. High Online Return Rates

- Major contributor to revenue loss

- Increases logistics, handling, and customer service costs

2. Revenue Concentration Risk

- Heavy dependence on a few countries

- Potential exposure to regional disruptions

3. Low Online Customer Visibility

- High share of anonymous and one-time buyers

- Limits customer lifetime value and retention strategies

Optimization Opportunities
1. Product Strategy

- Adopt channel-specific assortments

- Restrict high-return products from online sales

- Promote stable, low-return products online

2. Returns Management

- Introduce stricter online return policies

- Improve product descriptions and sizing information

- Analyze high-return products for root causes

3. Customer Strategy

Encourage customer identification in online sales (accounts, loyalty)

Focus on converting online one-time buyers into repeat customers

Leverage offline customer data for retention programs

4. Geographic Strategy

Tailor channel strategy by country

Prioritize offline-heavy strategies in core revenue markets

Pilot optimized online offerings in selective regions only